In [1]:
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from data.get_data import get_dataframe
from data_processor.calculate_stats import calculate_statistics
from data_processor.data_categorising import categories_columns
from data_processor.data_cleaner import clean_data
from data_processor.fight_stats import finalProcessingForFighter, calculateAverages
from data_processor.data_types_fixes import check_and_process_data_type, drop_col_for_training

In [2]:
og_df = get_dataframe('original.csv')

In [3]:
cleaned_df = clean_data(og_df)

In [4]:
stats_df = calculate_statistics(cleaned_df)

In [5]:
processed_df = finalProcessingForFighter(stats_df)

In [6]:
processed_df = check_and_process_data_type(processed_df)

In [7]:
avg_df = calculateAverages(processed_df)

In [8]:
cat_df = categories_columns(avg_df)

In [9]:
cat_df = categories_columns(avg_df)

In [10]:
df_for_training = drop_col_for_training(cat_df)

In [11]:
df_for_training = df_for_training.sort_index()
df_for_training

,Total_KD,Total_STR,Total_TD,Total_SUB,Opp_KD,Opp_STR,Opp_TD,Opp_SUB,Target,Avg_Round_Time,Avg_Round,Opp_Avg_Round_Time,Opp_Avg_Round,Weight_Class_code
0,0,0,0,0,0,0,0,0,0,0.000000,0.000000,0.0,0.0,8
1,0,0,0,0,0,0,0,0,0,0.000000,0.000000,170.0,1.0,8
2,0,0,0,0,0,0,0,0,0,0.000000,0.000000,747.0,1.0,8
3,0,0,0,0,0,0,0,0,0,0.000000,0.000000,591.0,1.0,8
4,0,0,0,0,0,0,0,0,0,0.000000,0.000000,454.5,1.5,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14819,1,64,0,0,0,0,0,0,0,309.666667,2.333333,0.0,0.0,5
14820,0,275,9,5,1,289,2,0,0,445.875000,3.750000,493.6,5.0,10
14821,0,76,0,0,0,226,6,2,0,300.000000,3.000000,486.5,5.5,13
14822,1,142,2,1,1,546,14,11,0,295.250000,3.250000,313.0,4.0,11


In [12]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn import metrics
import pandas as pd

X = df_for_training.drop("Target", axis=1)

In [13]:
y = df_for_training["Target"]

In [14]:
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [15]:
pipe = Pipeline([
    ("classifier", GradientBoostingClassifier(random_state=42))
])


In [16]:
param_grid = {
    "classifier__n_estimators": [100, 200, 400],
    "classifier__learning_rate": [0.05, 0.1],
    "classifier__max_depth": [2, 3],
}

In [17]:
grid = GridSearchCV(pipe, param_grid, cv=3, scoring="roc_auc", n_jobs=-1)

In [18]:
grid.fit(x_train, y_train)

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('classifier',
                                        GradientBoostingClassifier(random_state=42))]),
             n_jobs=-1,
             param_grid={'classifier__learning_rate': [0.05, 0.1],
                         'classifier__max_depth': [2, 3],
                         'classifier__n_estimators': [100, 200, 400]},
             scoring='roc_auc')

In [19]:
y_pred = grid.predict(x_test)

In [20]:
acc = metrics.accuracy_score(y_test, y_pred)
prec = metrics.precision_score(y_test, y_pred, zero_division=0)
rec = metrics.recall_score(y_test, y_pred, zero_division=0)
f1 = metrics.f1_score(y_test, y_pred, zero_division=0)


In [21]:
print("Best params:", grid.best_params_)
print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1:", f1)

Best params: {'classifier__learning_rate': 0.05, 'classifier__max_depth': 3, 'classifier__n_estimators': 200}
Accuracy: 0.5848758465011287
Precision: 0.5799065420560747
Recall: 0.569005043558001
F1: 0.5744040731312197


In [22]:
y_proba = grid.predict_proba(x_test)[:, 1]
roc_auc = float(metrics.roc_auc_score(y_test, y_proba))
print("ROC-AUC:", roc_auc)

ROC-AUC: 0.6181582766725605


In [23]:
importances = grid.best_estimator_.named_steps["classifier"].feature_importances_

In [24]:
feat_imp = pd.DataFrame({
    "feature": X.columns,
    "importance": importances
}).sort_values("importance", ascending=False)

feat_imp

,feature,importance
10,Opp_Avg_Round_Time,0.244393
1,Total_STR,0.157006
11,Opp_Avg_Round,0.116427
8,Avg_Round_Time,0.095864
2,Total_TD,0.083297
5,Opp_STR,0.079437
6,Opp_TD,0.047977
9,Avg_Round,0.037776
12,Weight_Class_code,0.036372
4,Opp_KD,0.036366
